## DS 3026 Final Project - Section 7

# Part 7 - Bias-Variance & Regularization

**Predictive Question:** Can we predict whether the home team will win an NBA game?

**Target Variable:** $Y ∈ {0, 1\\}$, where Y=1 if the home team wins and 0 otherwise 

**Model**: We compare a standard logistic regression model with a Lasso regression model and a Ridge regression model to analyze how model complexity affects performance. 

**Evaluation Metric:** We evaluate model performance using metrics such as `accuracy_score`, `precision_score`, `recall_score`, `f1_score`, and `roc_auc_score`, comparing training and test performance to assess overfitting and generalization. We also reviewed the coefficients of the features that the model utilized.

In [1]:
# imports 
import pandas as pd 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
# load game data 
games = pd.read_csv('./data/game_df.csv')

## 1 - Cleaning & Splitting 

Before the models are trained, we performed the same cleaning process on the columns as in Section 6. Creating rolling average and the difference columns helps our models be more accurate in predicting 'future' games. 

In [3]:
# dropping columns that are not needed for predictive modeling
cols_drop = ['min', 'team_abbreviation_home', 'team_abbreviation_away', 'team_nickname_home', 'team_nickname_away', 
             'team_city_name_home', 'team_city_name_away', 'pts_qtr1_home', 'pts_qtr2_home', 'pts_qtr3_home', 'pts_qtr4_home',
             'pts_ot1_home', 'pts_ot2_home', 'pts_ot3_home', 'pts_ot4_home', 'pts_ot5_home', 'pts_ot6_home', 'pts_ot7_home', 
             'pts_ot8_home', 'pts_ot9_home', 'pts_ot10_home', 'pts_qtr1_away', 'pts_qtr2_away', 'pts_qtr3_away', 'pts_qtr4_away', 
             'pts_ot1_away', 'pts_ot2_away', 'pts_ot3_away', 'pts_ot4_away', 'pts_ot5_away', 'pts_ot6_away', 'pts_ot7_away', 
             'pts_ot8_away', 'pts_ot9_away', 'pts_ot10_away', 'largest_lead_home', 'largest_lead_away', 'lead_changes', 'times_tied', 
             'plus_minus_home', 'plus_minus_away', 'video_available_home', 'video_available_away', 'season_type', 'team_rebounds_home',
             'team_rebounds_away', 'team_turnovers_home', 'total_turnovers_home', 'team_turnovers_away', 'total_turnovers_away', 
             'matchup_home', 'matchup_away', 'game_date', 'team_wins_losses_home', 'team_wins_losses_away']
games2 = games.drop(columns=cols_drop)

In [4]:
# change 'wl_home' to be numeric, with W = 1 and L = 0
games2['wl_home'] = games2['wl_home'].map({'W': 1, 'L': 0})

In [5]:
# columns that need to be transformed into rolling averages 
stat_home_cols = ['fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home', 
             'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home', 'reb_home', 
             'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home', 'pts_home',
             'pts_paint_home', 'pts_2nd_chance_home', 'pts_fb_home', 'pts_off_to_home']

stat_away_cols = ['fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away', 'fg3_pct_away',
             'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away', 'dreb_away', 'reb_away',
             'ast_away', 'stl_away', 'blk_away', 'tov_away', 'pf_away', 'pts_away', 
             'pts_paint_away', 'pts_2nd_chance_away', 'pts_fb_away', 'pts_off_to_away']

# for each stat column, create a new column that is the rolling average for the last 10 games
# home columns 
for col in stat_home_cols:
    games2[col + '_avg_10'] = games2.groupby('team_id_home')[col].transform(lambda x: x.rolling(10, min_periods=1).mean())
    games2 = games2.drop(columns=[col])

# away columns 
for col in stat_away_cols:
    games2[col + '_avg_10'] = games2.groupby('team_id_away')[col].transform(lambda x: x.rolling(10, min_periods=1).mean())
    games2 = games2.drop(columns=[col])

# verify new columns
games2.columns 

Index(['season_id', 'team_id_home', 'team_name_home', 'game_id', 'wl_home',
       'team_id_away', 'team_name_away', 'wl_away', 'game_date_est',
       'game_sequence', 'team_city_home', 'team_city_away', 'fgm_home_avg_10',
       'fga_home_avg_10', 'fg_pct_home_avg_10', 'fg3m_home_avg_10',
       'fg3a_home_avg_10', 'fg3_pct_home_avg_10', 'ftm_home_avg_10',
       'fta_home_avg_10', 'ft_pct_home_avg_10', 'oreb_home_avg_10',
       'dreb_home_avg_10', 'reb_home_avg_10', 'ast_home_avg_10',
       'stl_home_avg_10', 'blk_home_avg_10', 'tov_home_avg_10',
       'pf_home_avg_10', 'pts_home_avg_10', 'pts_paint_home_avg_10',
       'pts_2nd_chance_home_avg_10', 'pts_fb_home_avg_10',
       'pts_off_to_home_avg_10', 'fgm_away_avg_10', 'fga_away_avg_10',
       'fg_pct_away_avg_10', 'fg3m_away_avg_10', 'fg3a_away_avg_10',
       'fg3_pct_away_avg_10', 'ftm_away_avg_10', 'fta_away_avg_10',
       'ft_pct_away_avg_10', 'oreb_away_avg_10', 'dreb_away_avg_10',
       'reb_away_avg_10', 'ast_away_a

In [6]:
# create new difference df for the stats 
diff_df = games2[['game_id', 'team_id_home', 'team_id_away', 'game_date_est', 'wl_home']].copy() 

# create difference columns for each above stat
diff_df['fgm_dff'] = games2['fgm_home_avg_10'] - games2['fgm_away_avg_10']
diff_df['fga_diff'] = games2['fga_home_avg_10'] - games2['fga_away_avg_10']
diff_df['fg_pct_diff'] = games2['fg_pct_home_avg_10'] - games2['fg_pct_away_avg_10']
diff_df['fg3m_diff'] = games2['fg3m_home_avg_10'] - games2['fg3m_away_avg_10']
diff_df['fg3a_diff'] = games2['fg3a_home_avg_10'] - games2['fg3a_away_avg_10']
diff_df['fg3_pct_diff'] = games2['fg3_pct_home_avg_10'] - games2['fg3_pct_away_avg_10']
diff_df['ftm_diff'] = games2['ftm_home_avg_10'] - games2['ftm_away_avg_10']
diff_df['fta_diff'] = games2['fta_home_avg_10'] - games2['fta_away_avg_10']
diff_df['ft_pct_diff'] = games2['ft_pct_home_avg_10'] - games2['ft_pct_away_avg_10']
diff_df['oreb_diff'] = games2['oreb_home_avg_10'] - games2['oreb_away_avg_10']
diff_df['dreb_diff'] = games2['dreb_home_avg_10'] - games2['dreb_away_avg_10']
diff_df['reb_diff'] = games2['reb_home_avg_10'] - games2['reb_away_avg_10']
diff_df['ast_diff'] = games2['ast_home_avg_10'] - games2['ast_away_avg_10']
diff_df['stl_diff'] = games2['stl_home_avg_10'] - games2['stl_away_avg_10']
diff_df['blk_diff'] = games2['blk_home_avg_10'] - games2['blk_away_avg_10']
diff_df['tov_diff'] = games2['tov_home_avg_10'] - games2['tov_away_avg_10']
diff_df['pf_diff'] = games2['pf_home_avg_10'] - games2['pf_away_avg_10']
diff_df['pts_diff'] = games2['pts_home_avg_10'] - games2['pts_away_avg_10']
diff_df['pts_paint_diff'] = games2['pts_paint_home_avg_10'] - games2['pts_paint_away_avg_10']
diff_df['pts_2nd_chance_diff'] = games2['pts_2nd_chance_home_avg_10'] - games2['pts_2nd_chance_away_avg_10']
diff_df['pts_fb_diff'] = games2['pts_fb_home_avg_10'] - games2['pts_fb_away_avg_10']
diff_df['pts_off_to_diff'] = games2['pts_off_to_home_avg_10'] - games2['pts_off_to_away_avg_10']

diff_df 

,game_id,team_id_home,team_id_away,game_date_est,wl_home,fgm_dff,fga_diff,fg_pct_diff,fg3m_diff,fg3a_diff,...,ast_diff,stl_diff,blk_diff,tov_diff,pf_diff,pts_diff,pts_paint_diff,pts_2nd_chance_diff,pts_fb_diff,pts_off_to_diff
0,29600012,1610612747,1610612756,1996-11-01 00:00:00,1,-1.000000,-28.000000,0.140000,3.000000,5.000000,...,5.000000,-5.0,4.000000,11.000000,1.000000,14.000000,2.000000,8.000000,-11.000000,NaN
1,29600005,1610612748,1610612737,1996-11-01 00:00:00,1,10.000000,13.000000,0.064000,6.000000,5.000000,...,13.000000,-2.0,-3.000000,-5.000000,5.000000,13.000000,0.000000,-6.000000,-8.000000,NaN
2,29600002,1610612751,1610612739,1996-11-01 00:00:00,0,-11.000000,-14.000000,-0.075000,4.000000,8.000000,...,-3.000000,-3.0,6.000000,7.000000,-5.000000,-13.000000,10.000000,-2.000000,2.000000,NaN
3,29600007,1610612765,1610612754,1996-11-01 00:00:00,1,-1.000000,-6.000000,0.025000,0.000000,-1.000000,...,-7.000000,4.0,-1.000000,1.000000,-8.000000,6.000000,4.000000,-3.000000,-3.000000,NaN
4,29600013,1610612744,1610612746,1996-11-01 00:00:00,0,-14.000000,-11.000000,-0.117000,3.000000,10.000000,...,-1.000000,1.0,1.000000,0.000000,-10.000000,-12.000000,10.000000,10.000000,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28256,42200401,1610612743,1610612748,2023-06-01 00:00:00,1,-0.200000,-1.800000,0.005700,-1.600000,-4.300000,...,0.700000,-0.3,0.600000,0.100000,-0.700000,3.500000,0.400000,-1.200000,5.800000,-5.900000
28257,42200402,1610612743,1610612748,2023-06-04 00:00:00,0,0.700000,-2.700000,0.021500,-1.600000,-4.700000,...,0.800000,0.1,0.000000,0.400000,-1.700000,5.300000,3.000000,-0.400000,7.500000,-6.500000
28258,42200403,1610612748,1610612743,2023-06-07 00:00:00,0,-5.000000,-0.400000,-0.055600,2.700000,7.000000,...,-4.000000,1.1,0.200000,-0.500000,-1.000000,-6.800000,-6.800000,1.700000,-1.100000,0.400000
28259,42200404,1610612748,1610612743,2023-06-09 00:00:00,0,-4.500000,-1.200000,-0.046500,1.100000,3.900000,...,-4.300000,0.2,-0.100000,0.700000,-0.100000,-7.300000,-7.000000,0.300000,-1.000000,-1.000000


In [7]:
# creating train and test set, by date, using 75% of dates for training  
train = diff_df[diff_df['game_date_est'] < '2018-01-01']
test = diff_df[diff_df['game_date_est'] >= '2018-01-01']

# creating X as game data and y as win or loss for HOME TEAM
X_train = train.drop(columns=['game_id', 'team_id_home', 'team_id_away', 'game_date_est', 'wl_home']).dropna()
y_train = train.loc[X_train.index, 'wl_home']
X_test = test.drop(columns=['game_id', 'team_id_home', 'team_id_away', 'game_date_est', 'wl_home']).dropna()
y_test = test.loc[X_test.index, 'wl_home']

# verifying that X and y are same shape 
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (20159, 22)
y_train: (20159,)
X_test: (6133, 22)
y_test: (6133,)


## 2 - Models 

For the comparison of models, we looked at Logistic Regression, Lasso, and Ridge models. Before training the models, we did scale X_train and X_test to ensure the model was treating features fairly. 

In [8]:
# scale the data 
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
    columns=X_train.columns, index=X_train.index)

X_test_scaled = pd.DataFrame(scaler.transform(X_test),
    columns=X_test.columns, index=X_test.index)

In [9]:
# train logistic regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# and Lasso model w C=0.01
lasso_model = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, C=0.01)
lasso_model.fit(X_train_scaled, y_train)

# and Ridge model 
ridge_model = LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000)
ridge_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, solver='liblinear')

In [10]:
# make predictions for all models
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

y_pred_lasso = lasso_model.predict(X_test_scaled)
y_prob_lasso = lasso_model.predict_proba(X_test_scaled)[:, 1]

y_pred_ridge = ridge_model.predict(X_test_scaled)
y_prob_ridge = ridge_model.predict_proba(X_test_scaled)[:, 1]

After all 3 models were trained and made predictions, we calculated their prediction error metrics to compare the models. All of the models had extremely similar error metrics. The Logistic Regression model slightly outperformed the other two in terms of the accuracy and precision scores, and the Lasso model slightly outperformed the other models in terms of the recall and f1 scores. 

In [11]:
# calculate prediction error metrics 
acc_lr = round(accuracy_score(y_test, y_pred_lr), 4)
prec_lr = round(precision_score(y_test, y_pred_lr), 4)
recall_lr = round(recall_score(y_test, y_pred_lr), 4)
f1_lr = round(f1_score(y_test, y_pred_lr), 4)
roc_auc_lr = round(roc_auc_score(y_test, y_prob_lr), 4)

acc_lasso = round(accuracy_score(y_test, y_pred_lasso), 4)
prec_lasso = round(precision_score(y_test, y_pred_lasso), 4)
recall_lasso = round(recall_score(y_test, y_pred_lasso), 4)
f1_lasso = round(f1_score(y_test, y_pred_lasso), 4)
roc_auc_lasso = round(roc_auc_score(y_test, y_prob_lasso), 4)

acc_ridge = round(accuracy_score(y_test, y_pred_ridge), 4)
prec_ridge = round(precision_score(y_test, y_pred_ridge), 4)
recall_ridge = round(recall_score(y_test, y_pred_ridge), 4)
f1_ridge = round(f1_score(y_test, y_pred_ridge), 4)
roc_auc_ridge = round(roc_auc_score(y_test, y_prob_ridge), 4)

print(f'Logistic Regression:\n Accuracy: {acc_lr}\n Precision: {prec_lr}\n Recall: {recall_lr}\n F1 Score: {f1_lr}\n ROC AUC: {roc_auc_lr}')
print(f'\nLasso Regression:\n Accuracy: {acc_lasso}\n Precision: {prec_lasso}\n Recall: {recall_lasso}\n F1 Score: {f1_lasso}\n ROC AUC: {roc_auc_lasso}')
print(f'\nRidge Regression:\n Accuracy: {acc_ridge}\n Precision: {prec_ridge}\n Recall: {recall_ridge}\n F1 Score: {f1_ridge}\n ROC AUC: {roc_auc_ridge}')

Logistic Regression:
 Accuracy: 0.7094
 Precision: 0.7365
 Recall: 0.763
 F1 Score: 0.7495
 ROC AUC: 0.7714

Lasso Regression:
 Accuracy: 0.7093
 Precision: 0.7338
 Recall: 0.7685
 F1 Score: 0.7507
 ROC AUC: 0.7711

Ridge Regression:
 Accuracy: 0.708
 Precision: 0.735
 Recall: 0.7622
 F1 Score: 0.7483
 ROC AUC: 0.7714


Furthermore, looking at the coefficients for each feature generated by the 3 models, there are some differences in which features each model deemed the most important and necessary for prediction. One example is the `pts_diff` feature, which was calculated from the actual point scores from each game. The Logistic Regression and Ridge models had coefficients of 0.329 and 0.447 respectively, whereas Lasso had decreased its coefficient fully down to 0.0. 

In [12]:
coef_lr = pd.Series(lr_model.coef_[0], index=X_train_scaled.columns)
coef_lasso = pd.Series(lasso_model.coef_[0], index=X_train_scaled.columns)
coef_ridge = pd.Series(ridge_model.coef_[0], index=X_train_scaled.columns)

comparison = pd.DataFrame({
    "Logistic": coef_lr,
    "Lasso": coef_lasso,
    "Ridge": coef_ridge,
})

print("Coefficient Comparison (sorted by absolute value of Lasso coefficients):\n")
print(comparison.sort_values(by="Lasso", key=abs, ascending=True))

Coefficient Comparison (sorted by absolute value of Lasso coefficients):

                     Logistic     Lasso     Ridge
fgm_dff              0.136910  0.000000  0.143543
pts_2nd_chance_diff -0.009587  0.000000 -0.009997
pts_diff             0.329368  0.000000  0.446847
oreb_diff            0.268176  0.000000  0.268658
fta_diff            -0.308561  0.000000 -0.266439
pts_off_to_diff      0.016634  0.000000  0.016164
fg3a_diff           -0.130506  0.000000 -0.128993
dreb_diff            0.273777  0.017517  0.273383
pts_fb_diff         -0.058936 -0.034036 -0.059279
pf_diff             -0.047644 -0.038564 -0.047475
blk_diff             0.079000  0.059719  0.079137
ftm_diff             0.220523  0.062170  0.122021
fg3_pct_diff         0.006038  0.065877  0.007685
ast_diff             0.120387  0.096333  0.120484
pts_paint_diff      -0.098717 -0.099222 -0.098805
ft_pct_diff          0.017632  0.099913  0.034445
fg3m_diff            0.318270  0.224337  0.277722
stl_diff             0.389

**Bias-Variance Explanation:** The unregularized model may achieve strong performance on the training data but risks overfitting, leading to higher variance and reduced performance on unseen data. Regularization introduces a penalty that shrinks coefficient values, reducing variance at the cost of slightly increased bias.

**Effect of Regularization:** The regularized model produces more stable and generalizable predictions by preventing large coefficient estimates. This improves performance on the test set, even if training accuracy is slightly lower.

**Result:** In terms of performance, all three models performed similarly with accuracy scores around 0.706 to 0.708 range, AUCs around 0.767, and F1 scores around 0.75. Regularization didn't drastically change performance, neither the L1 or L2. 

However, in terms of accuracy, recall, and F1, Lasso (L1) Regression performed the best. The L1 regualization had the highest accuracy (0.7076), highest recall score (0.7784), and highest F1 score (0.7536). Regular Logistic Regression received the lowest accuracy score (0.7058), lowest recall score (0.7721), and lowest F1 score (0.751). The difference in ultimate performance is very slight, however. 

In the end, Logistic regression, which keeps all features, is more prone to higher variance, lasso regression sets many coefficients to 0 and has a lower variance, and ridge regression shrinks coefficients closer to 0 but not to 0 with moderate variance reduction.